# 02 — Morelos Local PDF Exploration
## Mexico Equality & Cyber Literacy Atlas · LGBTIQ+ Rights Compliance Pipeline

This notebook explores the **local fallback PDF documents** for Morelos stored
in `data/morelos/`. These are laws that were not automatically discoverable via
Orden Jurídico Nacional in notebook 01.

**Goals:**
- Inventory every PDF file present locally.
- Extract raw text with `pdfplumber`.
- Check which compliance-marker keywords appear in the extracted text.
- Identify which markers (L1–L3, C1–C3) have at least one candidate document
  and which still have no keyword match.

**Scope:** Text extraction and keyword presence only — no embeddings, no scoring yet.

---
## 1 · Setup & Imports

Standard dependencies only — no network or API calls in this notebook.

Install if needed:
```
pip install -r requirements.txt
```

In [1]:
import re
import unicodedata
import pathlib

import pandas as pd
import pdfplumber

# ── Local PDF directory ───────────────────────────────────────────────────────
DATA_DIR = pathlib.Path("data/morelos")

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"'{DATA_DIR}' not found. "
        "Place the Morelos PDF files in that directory before running this notebook."
    )

print(f"pdfplumber  : {pdfplumber.__version__}")
print(f"pandas      : {pd.__version__}")
print(f"Data dir    : {DATA_DIR.resolve()}")

pdfplumber  : 0.11.9
pandas      : 2.2.3
Data dir    : C:\Users\deaqu\OneDrive - Universidad Politécnica de Yucatán\Documentos\Carrera Academica\Data Engineer\7_Quadrimester\Internship\Validition-Pipeline-RAG-Law-and-Compliance-LGBTIQ-Rights\data\morelos


---
## 2 · Local File Inventory

Scan `data/morelos/` for all `.pdf` files and build a summary DataFrame.
Non-PDF files in the directory are listed but excluded from further processing.

In [2]:
all_files  = sorted(DATA_DIR.iterdir())
pdf_paths  = [f for f in all_files if f.suffix.lower() == ".pdf"]
other_files = [f for f in all_files if f.suffix.lower() != ".pdf"]

if other_files:
    print(f"Non-PDF files ignored ({len(other_files)}): "
          + ", ".join(f.name for f in other_files))

if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in '{DATA_DIR.resolve()}'.")

# ── Inventory DataFrame ───────────────────────────────────────────────────────
inventory_rows = [
    {
        "file_name": p.name,
        "file_path": str(p.resolve()),
        "size_kb":   round(p.stat().st_size / 1024, 1),
    }
    for p in pdf_paths
]

files_df = pd.DataFrame(inventory_rows)

print(f"\nPDF files found: {len(files_df)}\n")
files_df


PDF files found: 5



,file_name,file_path,size_kb
0,CONSTMOR.pdf,C:\Users\deaqu\OneDrive - Universidad Politécn...,3000.7
1,CPENALEM.pdf,C:\Users\deaqu\OneDrive - Universidad Politécn...,2793.5
2,LCDERHUMEM.pdf,C:\Users\deaqu\OneDrive - Universidad Politécn...,5200.4
3,LMENTALEM.pdf,C:\Users\deaqu\OneDrive - Universidad Politécn...,884.8
4,LSALUDEM.pdf,C:\Users\deaqu\OneDrive - Universidad Politécn...,1731.9


---
## 3 · Text Extraction per PDF

`pdfplumber` opens each PDF and extracts text page by page.

**Notes on extraction quality:**
- Digitally-created PDFs yield clean text immediately.
- Scanned / image-based PDFs return empty or near-empty text — those are
  flagged and would need OCR (`pytesseract`) in a future step before scoring.
- A 500-character preview is printed per file to assess quality at a glance.

In [3]:
PREVIEW_CHARS = 500


def extract_text(pdf_path: pathlib.Path) -> tuple[str, int]:
    """
    Extract all text from *pdf_path* using pdfplumber.
    Returns (full_text, page_count).
    Returns ("", 0) on error or if the PDF is image-only.
    """
    try:
        with pdfplumber.open(pdf_path) as pdf:
            page_count = len(pdf.pages)
            full_text  = "\n".join(page.extract_text() or "" for page in pdf.pages)
        return full_text, page_count
    except Exception as exc:
        print(f"  [ERROR] {pdf_path.name}: {exc}")
        return "", 0


# ── Extract text for every PDF ────────────────────────────────────────────────
# Result: {file_name: {"text": str, "page_count": int, "char_count": int}}
extracted: dict[str, dict] = {}

for pdf_path in pdf_paths:
    print(f"\n{'=' * 60}")
    print(f"File   : {pdf_path.name}")

    text, pages = extract_text(pdf_path)
    chars = len(text.strip())

    extracted[pdf_path.name] = {"text": text, "page_count": pages, "char_count": chars}

    print(f"Pages  : {pages}")
    print(f"Chars  : {chars}")

    if chars == 0:
        print("Preview: [no text extracted -- likely a scanned/image PDF]")
    else:
        preview = text.strip()[:PREVIEW_CHARS].replace("\n", " ")
        print(f"Preview: {preview}...")

print(f"\n{'=' * 60}")
print(f"Extraction complete: {len(extracted)} file(s) processed.")


File   : CONSTMOR.pdf
Pages  : 386
Chars  : 1256565
Preview: Constitución Política del Estado Libre y Soberano de Morelos, que reforma la del año de 1888 Consejería Jurídica del Poder Ejecutivo del Estado de Morelos. Última Reforma: 27-01-2026 Dirección General de Legislación. Subdirección de Jurismática. CONSTITUCIÓN POLÍTICA DEL ESTADO LIBRE Y SOBERANO DE MORELOS, QUE REFORMA LA DEL AÑO DE 1888 OBSERVACIONES GENERALES.- El Bando solemne en que se publicó la Constitución incluye en el nombre de la misma la leyenda "Que reforma la del año de 1888" pero en...

File   : CPENALEM.pdf
Pages  : 355
Chars  : 966572
Preview: Código Penal para el Estado de Morelos Consejería Jurídica del Poder Ejecutivo del Estado de Morelos. Última Reforma: 18-03-2026 Dirección General de Legislación. Subdirección de Jurismática. CÓDIGO PENAL PARA EL ESTADO DE MORELOS OBSERVACIONES GENERALES.- El artículo segundo transitorio deroga al Código Penal para el Estado Libre y Soberano de Morelos, No. 1178 Sección 

---
## 4 · Keyword Matching

Apply the same normalized keyword list used in notebook 01 against the **full
extracted text** of each PDF — not just the title.

Matching against the full text catches laws whose title doesn't surface the
keyword but whose body does (e.g. a *Código Penal* containing *discriminación*
in Article 287-bis).

The result DataFrame has one row per PDF with:
- `matched_keywords` — comma-separated list of found keywords
- `page_count` — number of pages
- `char_count` — total characters extracted (`0` signals a likely scanned PDF)

In [4]:
# ── Keyword list (identical to notebook 01) ───────────────────────────────────
KEYWORDS: list[str] = [
    # L1 / C2 -- non-discrimination & equality
    "discriminacion",
    "no discriminacion",
    "igualdad",
    # L2 -- gender identity recognition
    "identidad de genero",
    "cambio de nombre",
    "registro civil",
    # L3 -- same-sex unions / marriage
    "matrimonio",
    "sociedad de convivencia",
    "union civil",
    "concubinato",
    # C1 -- criminal provisions / hate crimes / violence
    "codigo penal",
    "violencia",
    "crimen de odio",
    "agravante",
    # C2 -- human-rights enforcement bodies
    "derechos humanos",
    "comision estatal",
    # C3 -- healthcare access
    "salud",
    "servicios de salud",
    "atencion medica",
    # General LGBTIQ+ scope
    "familia",
    "orientacion sexual",
    "diversidad sexual",
    "lgbtq",
    "lgbtiq",
    "genero",
]


def _normalize(text: str) -> str:
    """Lower-case and strip accents for robust keyword matching."""
    nfkd = unicodedata.normalize("NFKD", text)
    ascii_text = nfkd.encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", ascii_text).strip().lower()


def match_keywords(text: str) -> list[str]:
    """Return all KEYWORDS found anywhere in the normalized text."""
    norm = _normalize(text)
    return [kw for kw in KEYWORDS if kw in norm]


# ── Build keyword-match DataFrame ─────────────────────────────────────────────
kw_rows: list[dict] = []

for file_name, data in extracted.items():
    if data["char_count"] > 0:
        hits = match_keywords(data["text"])
    else:
        hits = []

    kw_rows.append({
        "file_name":        file_name,
        "matched_keywords": ", ".join(hits) if hits else "(none -- check for scanned PDF)",
        "page_count":       data["page_count"],
        "char_count":       data["char_count"],
    })

kw_df = pd.DataFrame(kw_rows)

print(f"Keyword match results ({len(kw_df)} files):\n")
kw_df

Keyword match results (5 files):



,file_name,matched_keywords,page_count,char_count
0,CONSTMOR.pdf,"discriminacion, igualdad, registro civil, matr...",386,1256565
1,CPENALEM.pdf,"discriminacion, igualdad, identidad de genero,...",355,966572
2,LCDERHUMEM.pdf,"derechos humanos, comision estatal, familia, g...",49,136977
3,LMENTALEM.pdf,"discriminacion, no discriminacion, igualdad, v...",79,208611
4,LSALUDEM.pdf,"discriminacion, registro civil, matrimonio, co...",203,520112


---
## 5 · Summary — Marker Coverage

Map the keyword hits back to the six compliance markers to answer:
*"Which markers have at least one candidate document in this folder,
and which still have no evidence at all?"*

| Marker | Description | Indicator keywords |
|--------|-------------|--------------------|
| **L1** | Non-discrimination (SO/GI) | discriminacion, igualdad, orientacion sexual, identidad de genero |
| **L2** | Gender-identity recognition | identidad de genero, cambio de nombre, registro civil |
| **L3** | Same-sex union / marriage | matrimonio, sociedad de convivencia, union civil |
| **C1** | Hate-crime / aggravated penalty | codigo penal, crimen de odio, agravante, violencia |
| **C2** | Anti-discrimination enforcement | derechos humanos, comision estatal, discriminacion |
| **C3** | Healthcare access | salud, servicios de salud, atencion medica |

A marker is **COVERED** if at least one PDF contains at least one of its
indicator keywords. This is a *necessary* condition, not sufficient — the
actual article text must be reviewed in notebook 05.

In [5]:
MARKER_KEYWORDS: dict[str, list[str]] = {
    "L1 - Non-discrimination (SO/GI)":     ["discriminacion", "igualdad", "orientacion sexual", "identidad de genero"],
    "L2 - Gender-identity recognition":     ["identidad de genero", "cambio de nombre", "registro civil"],
    "L3 - Same-sex union / marriage":       ["matrimonio", "sociedad de convivencia", "union civil"],
    "C1 - Hate-crime / aggravated penalty": ["codigo penal", "crimen de odio", "agravante", "violencia"],
    "C2 - Anti-discrimination enforcement": ["derechos humanos", "comision estatal", "discriminacion"],
    "C3 - Healthcare access":               ["salud", "servicios de salud", "atencion medica"],
}

# Pool all keywords found across every extractable PDF
all_found: set[str] = set()
for data in extracted.values():
    if data["char_count"] > 0:
        all_found.update(match_keywords(data["text"]))

# ── Marker coverage table ─────────────────────────────────────────────────────
covered:   list[str] = []
uncovered: list[str] = []

print("=" * 68)
print(f"  {'Marker':<40} {'Status':<9} Matched keywords")
print("=" * 68)

for marker, indicator_kws in MARKER_KEYWORDS.items():
    hits = [kw for kw in indicator_kws if kw in all_found]
    if hits:
        covered.append(marker)
        print(f"  {marker:<40} {'COVERED':<9} {', '.join(hits)}")
    else:
        uncovered.append(marker)
        print(f"  {marker:<40} {'MISSING':<9} --")

print("=" * 68)
print(f"\n  Covered : {len(covered)} / {len(MARKER_KEYWORDS)}")
print(f"  Missing : {len(uncovered)} / {len(MARKER_KEYWORDS)}")

if uncovered:
    print("\n  Markers with NO keyword evidence in any PDF:")
    for m in uncovered:
        print(f"    * {m}")
    print("\n  -> Add documents for these markers before running the scoring pipeline.")

# ── Scanned-PDF warning ───────────────────────────────────────────────────────
scanned = [name for name, d in extracted.items() if d["char_count"] == 0]
if scanned:
    print(f"\n  WARNING: {len(scanned)} PDF(s) yielded no extractable text (likely scanned):")
    for name in scanned:
        print(f"    * {name}")
    print("  -> These need OCR (pytesseract) before keyword matching is possible.")

  Marker                                   Status    Matched keywords
  L1 - Non-discrimination (SO/GI)          COVERED   discriminacion, igualdad, orientacion sexual, identidad de genero
  L2 - Gender-identity recognition         COVERED   identidad de genero, registro civil
  L3 - Same-sex union / marriage           COVERED   matrimonio
  C1 - Hate-crime / aggravated penalty     COVERED   codigo penal, agravante, violencia
  C2 - Anti-discrimination enforcement     COVERED   derechos humanos, comision estatal, discriminacion
  C3 - Healthcare access                   COVERED   salud, servicios de salud, atencion medica

  Covered : 6 / 6
  Missing : 0 / 6
